# 第 22 章 · 云任务：产物到本地的两道门

远端完成只意味着候选结果可取。还要回答：它属于哪个 task/attempt、基于什么版本、预览后本地有没有变化、应用后程序行为是否正确？

本章只实现产物入口：真实临时 Git 仓库、真实 `git apply` 和前后快照。没有云请求、保护沙箱或完整文件工具。输入源码是本章自建可信小函数，不执行外部代码。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · prepare 只计算候选后像

身份和哈希通过后，仍需检查 Git 基线与干净工作树，再在独立候选仓库试应用。预览不能偷偷修改目标。调整 `replacement` 可构造可应用但行为错误的补丁。

In [2]:
import difflib
from dataclasses import replace
m = load_example('22', 'artifact_gate')
run_dir = fresh_run_dir('ch22')
git_log = []
git = m.Git(git_log)
seed = 'def select(tasks, status):\n    return [t for t in tasks if t.get("state") == status]\n'
def fixture(name, replacement='status'):
    folder = run_dir / name
    repo = folder / 'repo'
    repo.mkdir(parents=True)
    (repo / 'task_list.py').write_text(seed)
    git.run(repo, 'init', '--quiet')
    git.run(repo, 'add', 'task_list.py')
    git.run(repo, '-c', 'user.name=Notebook Fixture', '-c', 'user.email=fixture@example.invalid',
            'commit', '--quiet', '-m', 'trusted baseline')
    base = git.run(repo, 'rev-parse', 'HEAD').stdout.decode().strip()
    updated = seed.replace('"state"', '"' + replacement + '"')
    patch = ('diff --git a/task_list.py b/task_list.py\n' + ''.join(difflib.unified_diff(
        seed.splitlines(True), updated.splitlines(True), 'a/task_list.py', 'b/task_list.py'))).encode()
    identity = dict(task_id='task-1', environment_id='local', attempt_id='attempt-1', result_revision='result-1')
    artifact = m.Artifact(**identity, base_commit=base, patch_sha256=m.digest(patch), patch=patch)
    return repo, m.ArtifactGate(git, repo, folder/'evidence', identity), artifact
repo, gate, artifact = fixture('normal')
prepared = gate.prepare(artifact)
assert prepared['phase'] == 'prepared'
assert (repo/'task_list.py').read_text() == seed
assert prepared['before'] != prepared['expected_after']
print({'phase': prepared['phase'], 'patch_sha256': artifact.patch_sha256})

{'phase': 'prepared', 'patch_sha256': 'b35e1492dca96879611ff975e51e9c2ba3a5c3ea3f94a51074f91bfd49b80ac4'}


## 2 · 应用成功之后才检查行为

文件后像匹配与业务正确是不同断言。这里的行为检查运行已知生成代码，便于观察两种结论；它不是允许执行任意云产物的沙箱。

In [3]:
def behavior(path):
    namespace = {}
    exec(compile(path.read_text(), str(path), 'exec'), namespace)
    tasks = [{'id': 'a', 'status': 'done'}, {'id': 'b', 'status': 'todo'}]
    return namespace['select'](tasks, 'done') == [tasks[0]]
applied = gate.commit()
assert applied['phase'] == 'applied' and behavior(repo/'task_list.py')
wrong_repo, wrong_gate, wrong_artifact = fixture('wrong-behavior', 'wrong_key')
wrong_gate.prepare(wrong_artifact)
assert wrong_gate.commit()['phase'] == 'applied'
assert not behavior(wrong_repo/'task_list.py')
print({'correct_patch': 'behavior_pass', 'applicable_wrong_patch': 'behavior_fail'})

{'correct_patch': 'behavior_pass', 'applicable_wrong_patch': 'behavior_fail'}


## 3 · 身份错误和预览过期都在写入前拒绝

哈希不证明来源，正确的来源也不保证工作树还没变。将本地新增文件改为编辑目标文件，快照检查仍应使旧预览失效。

In [4]:
bad_repo, bad_gate, bad_artifact = fixture('wrong-identity')
try:
    bad_gate.prepare(replace(bad_artifact, attempt_id='old-attempt'))
except m.Refused as error:
    assert str(error) == 'identity_mismatch:attempt_id'
else:
    raise AssertionError('错误身份未拒绝')
assert (bad_repo/'task_list.py').read_text() == seed
race_repo, race_gate, race_artifact = fixture('preview-race')
race_gate.prepare(race_artifact)
(race_repo/'reader-note.txt').write_text('预览后新增的本地工作')
try:
    race_gate.commit()
except m.Refused as error:
    assert str(error) == 'workspace_changed_since_preview'
else:
    raise AssertionError('过期预览未拒绝')
assert (race_repo/'task_list.py').read_text() == seed
print('身份拒绝和工作树变化拒绝均保留原文件')

身份拒绝和工作树变化拒绝均保留原文件


## 4 · 回包丢失后核对现场，不再 apply

模拟 Git 已写入、提交回执尚未落盘的窗口。恢复返回“观察到后像，仍须验证”，不会推断任务成功。取消若发生于 prepare 后，也可以直接放弃提交，此时目标应保持前像。

In [5]:
lost_repo, lost_gate, lost_artifact = fixture('lost-ack')
lost_gate.prepare(lost_artifact)
try:
    lost_gate.commit(lost_ack=True)
except m.Refused as error:
    assert str(error) == 'apply_ack_lost'
else:
    raise AssertionError('故障窗口未触发')
recovered = lost_gate.recover()
assert recovered['phase'] == 'applied_observed_requires_verification'
assert behavior(lost_repo/'task_list.py')
target_apply = [x for x in git_log if x['cwd'] == str(lost_repo) and x['argv'][1:2] == ['apply'] and '--check' not in x['argv']]
assert len(target_apply) == 1
try:
    lost_gate.commit()
except m.Refused as error:
    assert 'recover_first' in str(error)
else:
    raise AssertionError('恢复后不应盲目重放')
cancel_repo, cancel_gate, cancel_artifact = fixture('cancel-before-write')
cancel_gate.prepare(cancel_artifact)  # 不调用 commit，实际目标保持原样。
assert (cancel_repo/'task_list.py').read_text() == seed
print({'recovery': recovered['phase'], 'target_apply_calls': len(target_apply)})

{'recovery': 'applied_observed_requires_verification', 'target_apply_calls': 1}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/cloud-tasks/src/lib.rs#L213-L261](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/cloud-tasks/src/lib.rs#L213-L261)
- [codex-rs/cloud-tasks-client/src/http.rs#L340-L400](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/cloud-tasks-client/src/http.rs#L340-L400)
- [codex-rs/cloud-tasks-client/src/api.rs#L23-L175](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/cloud-tasks-client/src/api.rs#L23-L175)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。